<a href="https://colab.research.google.com/github/kerndutta-star/module3-ml-assignment/blob/main/Coding_Exercise_Prompt_Engineering_Prompt_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Exercise 1: Prompt Chaining for a Customer Support AI (Clean Output)
# Runs with a deterministic mock LLM (no API key needed).
# Optional: swap call_llm() to use OpenAI if you have a key.

import json
import textwrap

# -------------------------------
# Mock LLM (so it always runs)
# -------------------------------
def mock_llm(system_prompt: str, user_prompt: str) -> str:
    u = (system_prompt + "\n" + user_prompt).lower()

    # Step 1: classification JSON
    if "classify the customer's issue" in u or "return only valid json" in u:
        if "charged" in u or "refund" in u:
            return json.dumps({
                "category": "billing/refund",
                "urgency": "medium",
                "sentiment": "frustrated",
                "needs_escalation": False
            }, indent=2)
        if "password" in u or "login" in u:
            return json.dumps({
                "category": "account/login",
                "urgency": "high",
                "sentiment": "stressed",
                "needs_escalation": False
            }, indent=2)
        return json.dumps({
            "category": "general",
            "urgency": "low",
            "sentiment": "neutral",
            "needs_escalation": False
        }, indent=2)

    # Step 2: questions only
    if "ask for missing information" in u or "questions only" in u:
        return (
            "1) What email is linked to the account?\n"
            "2) What date(s) did you see the duplicate charge?\n"
            "3) What payment method was used?\n"
            "4) Do you see two receipts in your billing history?"
        )

    # Step 3: solution draft
    if "propose a solution draft" in u or "now propose a solution draft" in u:
        return textwrap.dedent("""
        - I can review the duplicate charge after account verification.
        - Please provide the account email and the charge date(s).
        - Check Billing > Transaction History and share invoice IDs or screenshots (hide full card numbers).
        - We will verify whether one charge is an authorization hold or a completed duplicate.
        - If it is a confirmed duplicate, we will follow the refund workflow and update you.

        Quick question: Can you share the account email and the two charge dates?
        """).strip()

    # Step 4: escalation JSON
    if "decide if we should escalate" in u or "escalation" in u:
        return json.dumps({
            "escalate": False,
            "reason": "Standard billing issue; can be handled by billing support."
        }, indent=2)

    return "OK"

# -------------------------------
# Call wrapper (mock by default)
# -------------------------------
def call_llm(system_prompt: str, user_prompt: str) -> str:
    return mock_llm(system_prompt, user_prompt)

# -------------------------------
# Prompt Chain
# -------------------------------
customer_message = "Hi, I was charged twice for my subscription and I can't find how to request a refund. Please help."

# Step 1: classify
SYSTEM_1 = "You are a customer support triage assistant. Be concise and strictly follow output format."
PROMPT_1 = f"""Classify the customer's issue.

Customer message:
{customer_message}

Return ONLY valid JSON with keys:
- category (one of: billing/refund, account/login, technical, shipping, general)
- urgency (low/medium/high)
- sentiment (calm/neutral/frustrated/angry)
- needs_escalation (true/false)

Return only JSON, no explanation.
"""

step1 = call_llm(SYSTEM_1, PROMPT_1)
triage = json.loads(step1)

# Step 2: ask missing info
SYSTEM_2 = "You are a helpful customer support agent. Friendly, efficient."
PROMPT_2 = f"""Ask for missing information needed to resolve this issue.

Triage info:
{json.dumps(triage, indent=2)}

Rules:
- Ask at most 4 questions.
- Questions only (no solutions or explanations).
- Friendly tone.
"""

step2 = call_llm(SYSTEM_2, PROMPT_2)

# Step 3: propose solution
SYSTEM_3 = "You are a support agent. Do not promise refunds or results. Provide clear, actionable next steps."
PROMPT_3 = f"""Customer message:
{customer_message}

Triage:
{json.dumps(triage, indent=2)}

Questions asked to customer (from prior step):
{step2}

Now propose a solution draft with at most 6 bullet points.
Rules:
- No promises (do not say 'you will get a refund').
- Mention verification steps if needed.
- End with one short question inviting reply.
"""

step3 = call_llm(SYSTEM_3, PROMPT_3)

# Step 4: escalation decision
SYSTEM_4 = "You are an escalation router. Output JSON only."
PROMPT_4 = f"""Given this triage info:
{json.dumps(triage, indent=2)}

Decide if we should escalate to a human specialist.
Escalate if:
- needs_escalation is true OR
- urgency is high and category is technical.

Return ONLY JSON with keys:
- escalate (true/false)
- reason (short string)
"""

step4 = call_llm(SYSTEM_4, PROMPT_4)

# -------------------------------
# Clean printing (UPDATED)
# -------------------------------
print("\n==============================")
print("EXERCISE 1: PROMPT CHAIN OUTPUT")
print("==============================")

print("\nSTEP 1 — Triage (JSON)")
print(json.dumps(triage, indent=2))

print("\nSTEP 2 — Questions to Customer")
# Convert "1) ..." style to "1. ..." style (cleaner)
for line in step2.splitlines():
    if line.strip():
        if ")" in line[:3]:
            num, rest = line.split(")", 1)
            print(f"{num.strip()}. {rest.strip()}")
        else:
            print(line)

print("\nSTEP 3 — Proposed Solution Draft")
print(step3)

print("\nSTEP 4 — Escalation Decision (JSON)")
print(step4)

# Optional: show prompts used for your submission doc
prompts_used = {
    "SYSTEM_1": SYSTEM_1, "PROMPT_1": PROMPT_1,
    "SYSTEM_2": SYSTEM_2, "PROMPT_2": PROMPT_2,
    "SYSTEM_3": SYSTEM_3, "PROMPT_3": PROMPT_3,
    "SYSTEM_4": SYSTEM_4, "PROMPT_4": PROMPT_4
}

print("\n--- Prompts used (copy into submission document) ---")
for k in ["SYSTEM_1","PROMPT_1","SYSTEM_2","PROMPT_2","SYSTEM_3","PROMPT_3","SYSTEM_4","PROMPT_4"]:
    print(f"\n{k}:\n{prompts_used[k]}")


EXERCISE 1: PROMPT CHAIN OUTPUT

STEP 1 — Triage (JSON)
{
  "category": "billing/refund",
  "urgency": "medium",
  "sentiment": "frustrated",
  "needs_escalation": false
}

STEP 2 — Questions to Customer
1. What email is linked to the account?
2. What date(s) did you see the duplicate charge?
3. What payment method was used?
4. Do you see two receipts in your billing history?

STEP 3 — Proposed Solution Draft
- I can review the duplicate charge after account verification.
- Please provide the account email and the charge date(s).
- Check Billing > Transaction History and share invoice IDs or screenshots (hide full card numbers).
- We will verify whether one charge is an authorization hold or a completed duplicate.
- If it is a confirmed duplicate, we will follow the refund workflow and update you.

Quick question: Can you share the account email and the two charge dates?

STEP 4 — Escalation Decision (JSON)
{
  "escalate": false,
  "reason": "Standard billing issue; can be handled by 